# Previsão de Volume de Incidentes — Albus Hub
### Challenge Locaweb/FIAP 2026 · Frente de Machine Learning (Integrante 2)

Este notebook conta a **história completa** do trabalho: do arquivo bruto até a previsão,
mostrando *o que* fizemos, *por que* fizemos e o número que sustenta cada decisão.

**Como ele se relaciona com o resto do projeto:**
- A **EDA** é feita aqui, célula a célula.
- A parte de **modelo** não é reimplementada: o notebook **importa** `pipeline_prioridades.py`,
  o mesmo arquivo que roda em produção. Assim o notebook nunca diverge do entregável.
- O documento de defesa completo é o `METODOLOGIA.md`; aqui é a versão executável.

⏱️ A seção 8 (rodar o pipeline) leva **2 a 4 minutos** — ela re-treina e refaz todo o backtest.

**Versão do modelo:** `volume_v3.2_2026-08-21`

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ML = os.getcwd()                      # o notebook mora na pasta ml_volume
if ML not in sys.path: sys.path.insert(0, ML)
CACHE = os.path.join(ML, "data", "incidents.parquet")

plt.rcParams.update({"figure.figsize":(13,4), "font.size":10, "axes.grid":True,
                     "grid.alpha":.25, "axes.spines.top":False, "axes.spines.right":False})
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)
print("dado tratado:", CACHE, "-", "existe" if os.path.exists(CACHE) else "NAO ENCONTRADO")

---
## 1. O problema

Prever **quantos incidentes serão abertos** amanhã (**D+1**) e daqui a uma semana (**D+7**),
por **prioridade** (P1–P5) e no total (ALL), entregando **um número e uma faixa de incerteza**.

É uma ferramenta de **apoio à decisão** — dimensionar plantão, antecipar pressão sobre times.
Não emite parecer.

---
## 2. A matéria-prima

O dataset da Locaweb tem **uma linha por incidente**. Antes de qualquer transformação, fizemos
backup do original com hash SHA-256 registrado — regra do projeto: **o bruto é fonte de verdade
e nunca é sobrescrito**. O que carregamos abaixo é o cache `.parquet` (mesmo conteúdo, abre
muito mais rápido).

In [ ]:
df = pd.read_parquet(CACHE)
df["Aberto"] = pd.to_datetime(df["Aberto"])
df["dia"]    = df["Aberto"].dt.floor("D")
df["_prio"]  = df["Prioridade"].str.extract(r"^\s*(\d)").astype("Int64")

print(f"linhas x colunas : {df.shape[0]:,} x {df.shape[1]}")
print(f"chave 'Numero'   : {'100% unica (sem duplicata)' if df['Número'].is_unique else 'TEM DUPLICATA'}")
print(f"periodo 'Aberto' : {df['Aberto'].min().date()} -> {df['Aberto'].max().date()}")
print("\ndistribuicao por ano:")
por_ano = df["Aberto"].dt.year.value_counts().sort_index()
print(pd.DataFrame({"linhas": por_ano, "%": (100*por_ano/len(df)).round(2)}).to_string())

### Decisão de limpeza nº 1 — usar só 2025

2023 e 2024 juntos somam poucas centenas de linhas em 730 dias: **cerca de 1 incidente por dia**.
Isso é período piloto, não operação — não forma série diária utilizável. 2025 concentra ~99% do
dado e tem **calendário completo** (365 de 365 dias com registro), então não há buraco para imputar.

In [ ]:
d25 = df[df["Aberto"].dt.year == 2025].copy()
print(f"2025: {len(d25):,} incidentes em {d25['dia'].nunique()}/365 dias do calendario")
print(f"descartado (2023+2024): {len(df)-len(d25):,} linhas "
      f"({100*(len(df)-len(d25))/len(df):.1f}% do total)")

---
## 3. Por que os dados oscilam

Encontramos **quatro fontes de oscilação diferentes** — e cada uma pede um tratamento diferente:

| Fonte | O que é | Como tratamos |
|---|---|---|
| **Patamar** | o degrau de 1º/set/2025 | avaliar só no regime novo (seção 3.1) |
| **Surto** | cascatas pai→filho | deduplicação (seção 4) |
| **Ciclo** | dia da semana, feriado | vira *feature* do modelo |
| **Ruído** | superdispersão de contagem | vai para o **intervalo**, não para o ponto |

### 3.1 O patamar — a quebra de regime de 1º de setembro

In [ ]:
idx25 = pd.date_range("2025-01-01", "2025-12-31")
serie_crua = d25.groupby("dia").size().reindex(idx25, fill_value=0)
corte = pd.Timestamp("2025-09-01")

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(serie_crua.index, serie_crua.values, lw=.9, color="#37474f")
ax.plot(serie_crua.index, serie_crua.rolling(7, center=True).mean(), lw=2.2, color="#c62828",
        label="média móvel 7 dias")
ax.axvline(corte, color="#1565c0", ls="--", lw=1.5)
ax.text(corte, ax.get_ylim()[1]*.95, "  1º/set — expansão do monitoramento",
        color="#1565c0", va="top", fontsize=9)
ax.set_title("Incidentes abertos por dia — 2025 (série crua)", fontweight="bold")
ax.set_ylabel("incidentes/dia"); ax.legend(); plt.show()

antes  = serie_crua[serie_crua.index <  corte].mean()
depois = serie_crua[serie_crua.index >= corte].mean()
print(f"jan-ago : {antes:6.1f} incidentes/dia")
print(f"set-dez : {depois:6.1f} incidentes/dia")
print(f"salto   : {depois/antes:.1f}x")

Um salto de ~7× de um dia para o outro. **A pergunta que decide todo o projeto:** a operação
piorou, ou só passamos a enxergar mais?

Para responder, decompomos o salto **por origem do incidente**.

In [ ]:
def taxa(mask, label):
    s = d25[mask].groupby("dia").size().reindex(idx25, fill_value=0)
    a, b = s[s.index < corte].mean(), s[s.index >= corte].mean()
    return {"recorte": label, "jan-ago/dia": round(a, 1), "set-dez/dia": round(b, 1),
            "razao": round(b/a, 2) if a > 0 else np.nan}

linhas = [
    taxa(d25["Aberto por"].eq("Manual"),        "Aberto por Manual"),
    taxa(d25["Aberto por"].eq("Monitoramento"), "Aberto por Monitoramento"),
    taxa(d25["Entrou para KPI?"].eq("SIM"),     "Elegivel a KPI"),
    taxa(pd.Series(True, index=d25.index),      "TOTAL"),
]
print(pd.DataFrame(linhas).to_string(index=False))

cis_ago = d25[d25["dia"].dt.month == 8]["Item de configuração"].nunique()
cis_set = d25[d25["dia"].dt.month == 9]["Item de configuração"].nunique()
print(f"\nCIs (ativos monitorados) distintos: agosto {cis_ago:,} -> setembro {cis_set:,} "
      f"(+{cis_set-cis_ago:,})")

**O veredito está na tabela:**

- Aberturas **manuais caíram** (razão < 1) — não há mais gente abrindo chamado.
- Aberturas por **monitoramento explodiram** (~11×).
- A carga **elegível a KPI ficou estável** (razão < 1) — o que o negócio mede não aumentou.
- Entraram ~1.700 **CIs novos** em setembro.

> 💡 **Insight de pitch:** o volume bruto multiplicou por ~7, mas a **carga operacional real
> ficou estável**. Quem olhar só o gráfico conclui "a operação está afundando" — e erra. O dado
> apenas ganhou visibilidade.

**Consequência metodológica:** são dois mundos. O modelo **treina** com os dois (mais dado é
melhor — testamos isso), mas a **nota** só pode sair no mundo em que ele vai operar (~750/dia).
Medir acerto em jan–ago daria um erro absoluto pequeno só porque o volume era pequeno.

---
## 4. Os picos são cascatas, não crises

Quando um ativo cai, o monitoramento não abre 1 incidente: abre **1 pai + dezenas ou centenas de
filhos** (cada serviço afetado vira um alarme). A coluna `Incidente Pai` marca quem é filho.

Vamos olhar o maior pico do P2 no ano.

In [ ]:
p2 = d25[d25["_prio"].eq(2)]
pico = p2.groupby("dia").size().idxmax()

sub    = p2[p2["dia"].eq(pico)]
filhos = sub["Incidente Pai"].notna().sum()
maior  = sub.loc[sub["Incidente Pai"].notna(), "Incidente Pai"].value_counts()

print(f"Dia de pico do P2 : {pico.date()}")
print(f"  total no dia     : {len(sub)}")
print(f"  eram filhos      : {filhos} ({100*filhos/len(sub):.0f}%)")
print(f"  maior cascata    : 1 unico incidente-pai gerou {maior.iloc[0]} filhos")
print(f"  eventos-RAIZ     : {len(sub)-filhos}   <- o numero real de problemas do dia")

### Decisão de limpeza nº 2 — deduplicar cascatas

Ficamos apenas com os incidentes **sem pai** (os "raiz"). Ou seja: **contar eventos, não alarmes.**

**Por que isso é legítimo e não "jogar dado fora":** o próprio dicionário de dados diz que
*incidente com `Incidente Pai` preenchido não entra no KPI*. Deduplicar é contar exatamente o
que o negócio já mede. Um pico de centenas de alarmes não é centenas de problemas — é **um**
problema.

In [ ]:
dedup = d25[d25["Incidente Pai"].isna()]

def diaria(frame, prio=None):
    f = frame if prio is None else frame[frame["_prio"].eq(prio)]
    return f.groupby("dia").size().reindex(idx25, fill_value=0).astype(float)

sl = slice("2025-09-01", "2025-12-31")
fig, axes = plt.subplots(2, 1, figsize=(13, 7), constrained_layout=True)
for ax, p, cor in zip(axes, [2, 3], ["#1565c0", "#ef6c00"]):
    ax.plot(diaria(d25, p)[sl].index,   diaria(d25, p)[sl].values,   lw=1,
            color="#c0c0c0", label=f"P{p} cru (alarmes)")
    ax.plot(diaria(dedup, p)[sl].index, diaria(dedup, p)[sl].values, lw=1.4,
            color=cor, label=f"P{p} deduplicado (eventos)")
    ax.set_title(f"P{p} — a deduplicação remove os storms mantendo o sinal de fundo")
    ax.set_ylabel("por dia"); ax.legend()
plt.show()

In [ ]:
def acf(s, k):
    s = np.asarray(s, float)
    return np.corrcoef(s[:-k], s[k:])[0, 1] if s.std() > 0 else np.nan

linhas = []
for p in [2, 3]:
    for frame, tag in [(d25, "cru"), (dedup, "dedup")]:
        s = diaria(frame, p)[sl]
        linhas.append({"serie": f"P{p} {tag}", "max/dia": int(s.max()),
                       "cv": round(s.std()/s.mean(), 2), "acf1": round(acf(s, 1), 2)})
print(pd.DataFrame(linhas).to_string(index=False))

**Leia a coluna `acf1`** — ela mede o quanto o dia de hoje ajuda a prever o de amanhã.

No P2 ela salta de ~0,05 (praticamente nada: série imprevisível) para ~0,32. E o `cv`
(irregularidade) cai pela metade.

> A deduplicação **não melhorou o modelo — ela revelou o sinal que os alarmes escondiam.**

---
## 5. Cada prioridade é um bicho diferente

Por isso treinamos **uma série por prioridade** em vez de um modelo único: elas não se parecem.

In [ ]:
series = {"ALL": diaria(dedup)}
for p in [1, 2, 3, 4, 5]:
    series[f"P{p}"] = diaria(dedup, p)

fig, axes = plt.subplots(3, 2, figsize=(14, 8), constrained_layout=True)
for ax, (nome, s) in zip(axes.ravel(), series.items()):
    ax.plot(s[sl].index, s[sl].values, lw=1, color="#37474f")
    ax.set_title(f"{nome} — média {s[sl].mean():.1f}/dia", fontsize=10)
fig.suptitle("Séries deduplicadas por prioridade (set–dez 2025)", fontweight="bold")
plt.show()

In [ ]:
dias = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
perfil = {}
for nome in ["ALL", "P2", "P3", "P4"]:
    s = series[nome][sl]
    g = s.groupby(s.index.dayofweek).mean()
    perfil[nome] = [g.get(k, np.nan) for k in range(7)]
P = pd.DataFrame(perfil, index=dias).T
P["fds/útil"] = (P[["sáb", "dom"]].mean(axis=1)
                 / P[["seg", "ter", "qua", "qui", "sex"]].mean(axis=1)).round(2)
print("Média por dia da semana:\n")
print(P.round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
for nome, cor in [("P2", "#1565c0"), ("P3", "#ef6c00"), ("P4", "#2e7d32")]:
    v = P.loc[nome, dias].astype(float)
    ax.plot(dias, v/v.mean(), marker="o", color=cor, label=nome)
ax.axhline(1, color="#b0bec5", ls="--")
ax.set_title("Perfil semanal normalizado (1,0 = média da própria série)", fontweight="bold")
ax.set_ylabel("relativo à média"); ax.legend(); plt.show()

In [ ]:
linhas = []
for nome in ["ALL", "P2", "P3", "P4", "P5"]:
    y     = series[nome][sl]
    tend  = y.rolling(7, center=True).mean()
    detr  = y - tend
    saz   = detr.groupby(detr.index.dayofweek).transform("mean")
    resid = (detr - saz).dropna()
    vr    = float(np.var(resid))
    Fsaz  = max(0.0, 1 - vr/max(1e-9, float(np.var(saz.reindex(resid.index) + resid))))
    Ftend = max(0.0, 1 - vr/max(1e-9, float(np.var(tend.reindex(resid.index) + resid))))
    linhas.append({"serie": nome, "media/dia": round(y.mean(), 1),
                   "cv": round(y.std()/y.mean(), 2), "var/media": round(y.var()/y.mean(), 1),
                   "F_sazonal": round(Fsaz, 2), "F_tendencia": round(Ftend, 2),
                   "acf1": round(acf(y, 1), 2), "acf7": round(acf(y, 7), 2)})
print(pd.DataFrame(linhas).to_string(index=False))

**Como ler:** `F_sazonal` e `F_tendencia` vão de 0 (não existe) a 1 (domina). `var/media` seria
**1** se fosse contagem pura (Poisson) — acima disso indica surtos. `acf7` mede o quanto o mesmo
dia da semana passada ajuda a prever.

**O que os números ensinam:**

- **P2 é humana** — cai ~metade no fim de semana e tem a maior força sazonal. Segue horário comercial.
- **P4 é máquina** — força sazonal ~0,03 e `acf7` ~0,00: totalmente plana, roda 24/7, **não sabe
  que é domingo**.
- **P3 é a mais volátil** e a que mais anda de nível (tendência alta).
- **Todas são fortemente superdispersas** (`var/media` de 30 a 60 contra 1 do Poisson): boa parte
  do movimento é **surto genuinamente imprevisível**. Esse é o teto de acerto.
- **P5 é intermitente** (quase sempre 0) e **P1 teve ~1 evento no ano** — não há série.

> ⚠️ **A descoberta que mudou a avaliação:** como o **P4 não tem sazonalidade semanal**, comparar
> o modelo contra a régua "mesmo dia da semana passada" ali é competir contra um espantalho. Foi
> isso que nos fez trocar o baseline (seção 7).

---
## 6. Como o modelo aprende

Transformamos previsão de série temporal em **regressão supervisionada**: uma linha por dia, a
resposta é o valor de D+1 (ou D+7), e as perguntas são pistas conhecidas de antemão.

| Grupo de features | Quais |
|---|---|
| Calendário (determinístico) | dia da semana, fim de semana, feriado BR, Black Friday, temporada de dezembro |
| Histórico | mesmo dia semana passada, último valor, média/desvio dos últimos 7 e 28 dias |
| Exposição | nº de CIs ativos — é o que "atravessa" a quebra de regime |
| Tendência | contador de tempo |

**Antivazamento (*data leakage*) — regra inegociável:**
- Split **temporal**: passado treina, futuro testa. **Nunca** aleatório.
- Toda feature de histórico leva `shift(h)`: ao prever D+7, o modelo só vê o que se sabia 7 dias antes.
- Calendário do dia-alvo **pode** ser usado (é determinístico — sabemos hoje que 25/12 é feriado).
- **Proibido** derivar qualquer coisa de `Resolvido`, `Encerrado`, `Duração`, `Código de fechamento`
  — isso só existe *depois* que o incidente acabou.

---
## 7. Os algoritmos e o protocolo de avaliação

Montamos uma **escada de complexidade** e só pagamos complexidade quando ela compra acerto.

| Degrau | Preditor | Por que está aqui |
|---|---|---|
| 0 | `naive7` — mesmo dia da semana passada | régua para série com ciclo semanal |
| 0 | `media7` — média dos últimos 7 dias | régua para série **sem** ciclo semanal |
| 0 | `ultimo` — repete o último valor | régua de persistência; imbatível em série lisa |
| 1 | **Ridge** (linear regularizada) | o **interpretável** — dá para ler o peso de cada pista |
| 2 | **Poisson-offset** | família correta para **contagem**; nunca prevê negativo |
| 3 | **GBR** (boosting) | capta não-linearidade; mede o teto. Caixa-preta |

**O que descartamos:** LSTM/redes neurais (só ~120 dias de regime pleno — decoraria e não é
interpretável) e ARIMA/Prophet como carro-chefe (absorvem pior as features externas).

### As duas correções de honestidade metodológica

1. **Baseline justo.** Antes comparávamos só com `naive7` — que a seção 5 mostrou ser um
   espantalho no P4. Agora o ganho é medido contra a **melhor das três réguas**.
2. **Preditor escolhido fora do período da nota:**

```
set-out/2025 (61 dias)  ->  escolhe o preditor    [nunca vira nota]
nov-dez/2025 (61 dias)  ->  produz a nota         [nunca participa da escolha]
```

E as réguas simples são **candidatas de pleno direito**: onde a régua ganha, **entregamos a
régua**. Isso é decisão de engenharia, não fracasso do ML.

---
## 8. Rodando o pipeline de verdade

A célula abaixo **importa** `pipeline_prioridades.py` — o mesmo arquivo que roda em produção.
Importar já executa todo o fluxo: features → backtest walk-forward → seleção → intervalo
conformal → previsão futura → gravação de `outputs/` e `plots/`.

**Por que importar em vez de copiar o código para cá:** se o notebook reimplementasse os modelos,
existiriam duas versões que podem divergir em silêncio. Assim há **uma fonte de verdade só**.

⏱️ **Leva de 2 a 4 minutos.**

In [ ]:
import pipeline_prioridades as pl   # executa o pipeline completo e imprime o resumo

### Como as features ficam na prática

Uma espiada na matriz que o modelo realmente vê (série P3, horizonte D+1):

In [ ]:
X, y = pl.build_Xy(pl.series["P3"], 1)
cols = ["seas_lag7", "last", "roll7_mean", "roll28_mean", "exp_roll7",
        "dow_0", "is_weekend", "is_holiday"]
amostra = X.loc["2025-11-03":"2025-11-09", cols].copy()
amostra.insert(0, "ALVO (real do dia)", y.loc["2025-11-03":"2025-11-09"])
print("Cada linha = um dia. O ALVO é o que queremos prever; o resto só usa o passado.\n")
print(amostra.round(1).to_string())

---
## 9. Resultados

Nota em **nov–dez/2025** (61 dias que não participaram da escolha), ganho medido contra a
**melhor** das três réguas bobas.

In [ ]:
linhas = []
for escopo in pl.series:
    for h in ["D+1", "D+7"]:
        m  = pl.all_metrics[escopo][h]
        sk = m["skill_vs_melhor_regua"]
        cb = m["_coverage"]["conformal_nov_dez"]
        linhas.append({
            "serie": escopo, "horiz": h,
            "preditor": m["_escolhido"], "tipo": m["_tipo"],
            "MAE": round(m["MAE"], 1), "sMAPE%": round(m["sMAPE"], 1),
            "skill vs melhor regua": f"{sk*100:+.0f}%" if np.isfinite(sk) else "n/a",
            "cobertura": f"{cb*100:.0f}%" if np.isfinite(cb) else "n/a",
        })
print(pd.DataFrame(linhas).to_string(index=False))

### Auditoria: e os outros candidatos?

O `metrics.json` guarda o MAE de **todos** os preditores, para qualquer um poder conferir que a
escolha não foi conveniente.

In [ ]:
for escopo, h in [("P2", "D+1"), ("P4", "D+1"), ("P3", "D+7")]:
    m = pl.all_metrics[escopo][h]
    tab = pd.DataFrame({
        "MAE set-out (escolha)": pd.Series(m["_mae_candidatos_out"]),
        "MAE nov-dez (nota)":    pd.Series(m["_mae_candidatos_nov_dez"]),
    })
    print(f"\n=== {escopo} {h} — escolhido: {m['_escolhido']} ===")
    print(tab.to_string())

In [ ]:
img = plt.imread(os.path.join(ML, "plots", "backtest_v32_D1.png"))
fig, ax = plt.subplots(figsize=(14, 13))
ax.imshow(img); ax.axis("off"); plt.show()

### A previsão que o time consome

Esta é a interface que o app / Power BI chama — sem abrir notebook.

In [ ]:
from predict import prever_volume
previsao = prever_volume()
print(previsao.to_string(index=False))
previsao          # no notebook, renderiza a tabela formatada

---
## 10. Conclusões

**Em D+7 o modelo ganha em todas as séries** (+13% a +30% sobre a melhor régua). Faz sentido
mecanicamente: a 7 dias de distância a regra "repete o valor de ontem" perde a validade, e é aí
que calendário, exposição e nível recente passam a valer.

**Em D+1 o modelo só ganha claramente no P2** (+16%). Em ALL e P4 — séries lisas e persistentes —
a regra trivial é imbatível, e o pipeline **entrega a regra trivial**.

**O intervalo conformal está calibrado**: cobertura entre 75% e 84% (meta 80%) em todas as séries.

### Mancha declarada

No **P3 D+1** a seleção escolheu GBR (−7%), mas o Ridge teria dado **+16%**. A escolha errou
porque 61 dias de seleção ainda é pouco. Preferir o Ridge aqui exigiria olhar o período da nota —
que é exatamente o vício que o protocolo existe para eliminar. Fica registrado como limitação.

### Sobre números de versões anteriores

O "+35%" que circulou antes era aritmeticamente correto, mas medido contra a régua fraca e com o
modelo escolhido olhando o próprio teste. **Os números deste notebook são menores e defensáveis.**

### Limitações

1. Regime pleno tem ~4 meses → captamos sazonalidade **semanal**, não anual.
2. Janela de seleção curta (61 dias) — causa da mancha do P3 D+1.
3. *Storms* de cascata são imprevisíveis no *timing* → vão para o intervalo, não para o ponto.
4. Previsões para 2026 **extrapolam** — o modelo nunca viu janeiro.
5. Séries superdispersas (`var/media` 30–60) têm teto de acerto que nenhum modelo vence.
6. Não avaliamos jan–ago com **erro relativo** (teste de robustez no regime antigo) — identificado
   como próximo passo, não executado.

### Próximos passos

- Avaliar jan–ago com erro relativo (adimensional) como teste de robustez.
- Croston (demanda intermitente) dedicado para P5, se o negócio quiser.
- Handoff para a frente de **risco/score** (Integrante 3): mesmo dado tratado, alvo `KPI Violado?`.
- Plugar `prever_volume()` no app do time / Power BI.

---
*Documento de defesa completo: `METODOLOGIA.md` · Resumo executivo: `README.md`*